# Imports

In [1]:
import importlib
import sys
import torch

sys.path.insert(0, '../..')
sys.path.insert(0, '../../../')
sys.path.insert(0, '../../../../../load/event_log_loader')

import new_event_log_loader

# Data

### Load Data Files

In [2]:
# Path to your pickle file (saved with torch.save)
file_path_train = '../../../../../load/encoded_data/BPIC_2019_all_1_train.pkl'
# Load the dataset using torch.load
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)
# Check the type of the loaded dataset
print(type(helpdesk_train_dataset))

# Path to your pickle file (saved with torch.save)
file_path_val = '../../../../../load/encoded_data/BPIC_2019_all_1_val.pkl'
# Load the dataset using torch.load
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)
# Check the type of the loaded dataset
print(type(helpdesk_val_dataset))


<class 'new_event_log_loader.EventLogDataset'>
<class 'new_event_log_loader.EventLogDataset'>


### Train Data Insights

In [3]:
# Helpdesk Dataset Categories, Features:
helpdesk_all_categories = helpdesk_train_dataset.all_categories

helpdesk_all_categories_cat = helpdesk_all_categories[0]
print(helpdesk_all_categories_cat)

helpdesk_all_categories_num = helpdesk_all_categories[1]
print(helpdesk_all_categories_num)

for i, cat in enumerate(helpdesk_all_categories_cat):
     print(f"Helpdesk (5) Categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk (5) Total Amount of Category labels: {cat[1]}")

print('\n')    

for i, num in enumerate(helpdesk_all_categories_num):
     print(f"Helpdesk (5) Numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk (5) Amount Numerical: {num[1]}")
     
# Get concept_name id:
# 
concept_name = 'concept:name_start'
concept_name_id = [i for i, cat in enumerate(helpdesk_all_categories[0]) if cat[0] == concept_name][0]

print("ID concet name in cat list: ", concept_name_id)

duration_seconds = 'duration_seconds'
duration_seconds_id = [i for i, num in enumerate(helpdesk_all_categories[1]) if num[0] == duration_seconds][0]
print("ID duration_seconds in num list: ", duration_seconds_id)

[('concept:name_start', 43, {'Block Purchase Order Item': 1, 'Cancel Goods Receipt': 2, 'Cancel Invoice Receipt': 3, 'Cancel Subsequent Invoice': 4, 'Change Approval for Purchase Order': 5, 'Change Currency': 6, 'Change Delivery Indicator': 7, 'Change Final Invoice Indicator': 8, 'Change Price': 9, 'Change Quantity': 10, 'Change Rejection Indicator': 11, 'Change Storage Location': 12, 'Change payment term': 13, 'Clear Invoice': 14, 'Create Purchase Order Item': 15, 'Create Purchase Requisition Item': 16, 'Delete Purchase Order Item': 17, 'Reactivate Purchase Order Item': 18, 'Receive Order Confirmation': 19, 'Record Goods Receipt': 20, 'Record Invoice Receipt': 21, 'Record Service Entry Sheet': 22, 'Record Subsequent Invoice': 23, 'Release Purchase Order': 24, 'Release Purchase Requisition': 25, 'Remove Payment Block': 26, 'SRM: Awaiting Approval': 27, 'SRM: Change was Transmitted': 28, 'SRM: Complete': 29, 'SRM: Created': 30, 'SRM: Deleted': 31, 'SRM: Document Completed': 32, 'SRM: He

In [4]:
selected_cat_attributes = ['concept:name_start', 'org:resource_start']
selected_num_attributes = ['seconds_in_day', 'day_in_week']

selected_categories = (
    [cat for cat in helpdesk_all_categories[0] if cat[0] in selected_cat_attributes],
    [num for num in helpdesk_all_categories[1] if num[0] in selected_num_attributes]
)

# Loss Object Creation

# Training Configuration

In [5]:
import stochasticLSTM.model

importlib.reload(stochasticLSTM.model)
from stochasticLSTM.model import StochasticLSTM

"""
Specific model parameters from paper: 
"""

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")

# Size hidden layer
hidden_size = 128

# Number of LSTM cells
num_layers = 2

# Fixed Dropout probability
p_fix = 0.1

# Lambda for L2 (weight, bias, dropout) regularization: According to formula: 1/2N
regularization_term = 1e-5

# Hans Weytjens LSTM model
model = StochasticLSTM(
    data_set_categories=helpdesk_all_categories,
    model_input_feat=selected_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    weight_reg=regularization_term,
    p_fix=p_fix,
    device=device,
)

import loss.losses

importlib.reload(loss.losses)
from loss.losses import Loss

loss_obj = Loss()


import training.train

importlib.reload(training.train)
from training.train import Training

from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(comment="train")


"""
Parameter of Probabilistic Suffix Prediction experimental design, to ensure fair comparison:
"""

# Start learning rate
learning_rate = 5e-3

# Optimizer and Scheduler
optimizer = torch.optim.Adam(
    params=model.parameters(), lr=learning_rate, weight_decay=0
)
scheduler = ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=20, min_lr=1e-10
)

# Epochs
num_epochs = 200

# Batch of model input
batch_size = 128

# shuffle data
shuffle = True

optimize_values = {
    "optimizer": optimizer,
    "scheduler": scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
}

trainer = Training(
    model=model,
    device=device,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    selected_features=(selected_cat_attributes, selected_num_attributes),
    concept_name_id=concept_name_id,
    duration_seconds_id=duration_seconds_id,
    loss_obj=loss_obj,
    optimize_values=optimize_values,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path="model.pkl",
)

# Train the model:
trainer.train()

Embeddings:  ModuleList(
  (0): Embedding(43, 16)
  (1): Embedding(610, 24)
)
Total embedding feature size:  40
Input feature size:  42
Cells hidden size:  128
Number of LSTM layer:  2
Dropout rate:  0.1




Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7f2be2044b20>
Epochs:  200
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/7015 [00:00<?, ?it/s]

Epoch [1/200], Learning Rate: 0.005
Training: Avg Attenuated Training Loss: 1.8544


...

Epoch [199/200], Learning Rate: 1.953125e-05
Training: Avg Attenuated Training Loss: 23.1595


Validation: Avg Standard Validation Loss: 0.6629
Validation: Avg Attenuated Validation Loss: 70.5139
Validation Loss for Scheduler: 0.6629
saving model


  0%|          | 0/7015 [00:00<?, ?it/s]

Epoch [200/200], Learning Rate: 1.953125e-05
Training: Avg Attenuated Training Loss: 11.9257


Validation: Avg Standard Validation Loss: 0.6629
Validation: Avg Attenuated Validation Loss: 22.8584
Validation Loss for Scheduler: 0.6629
saving model
Training complete.
Model saved to path: model.pkl
